# EVSE status time series (DuckDB)

This notebook analyses the monthly Parquet files in `parquet/` with [DuckDB](https://duckdb.org/).
They are built from the git history by `build_parquet.py` (see README) and combine both sources:

- `oicp`: legacy OICP status file (until the switch in November 2026)
- `ocpi`: snapshot of the new OCPI API (`data/ocpi/status.json`)

**`status_intervals_YYYY-MM.parquet`**: one row per EVSE and period with unchanged status, `[valid_from, valid_to)`.
Intervals are cut at month boundaries, so every file is self-contained.

| column | description |
|---|---|
| `source` | `oicp` or `ocpi` |
| `evse_id` | EVSE ID, e.g. `CH*SUI*E…` (identical in OICP `EvseID` and OCPI `evse_id`) |
| `operator_id` | e.g. `CH*SUI` |
| `location_id`, `evse_uid` | OCPI only |
| `status` | common vocabulary: `AVAILABLE`, `OCCUPIED`, `OUT_OF_SERVICE`, `UNKNOWN`, `REMOVED` |
| `status_raw` | original status of the source |
| `valid_from`, `valid_to` | observation time (git commit time, UTC) |
| `status_last_updated` | OCPI only: time of the status change reported by the CPO |

**`snapshots_YYYY-MM.parquet`**: one row per observed snapshot (`source`, `observed_at`, `commit`, `evse_count`), useful to check the coverage and gaps of the scraper.


In [ ]:
import os
import duckdb
import matplotlib.pyplot as plt

PARQUET_DIR = os.environ.get("PARQUET_DIR", "parquet")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(f"""
    CREATE OR REPLACE VIEW intervals_raw AS
    SELECT * FROM read_parquet('{PARQUET_DIR}/status_intervals_*.parquet');
    CREATE OR REPLACE VIEW snapshots AS
    SELECT * FROM read_parquet('{PARQUET_DIR}/snapshots_*.parquet');
""")
con.sql("""
    SELECT source, min(valid_from) AS first, max(valid_to) AS last,
           count(*) AS intervals, count(DISTINCT evse_id) AS evses
    FROM intervals_raw GROUP BY source ORDER BY first
""").df()

## Combined time series

During the transition both sources are available. The view `intervals` combines them to one continuous series:
OICP is used until `CUTOVER`, OCPI afterwards. By default `CUTOVER` is the last OICP snapshot
(i.e. OICP is used as long as it exists, because it covers more EVSEs during the migration).
Set `CUTOVER` to a timestamp string (e.g. `"2026-11-15 00:00:00+00"`) to switch earlier.

In [ ]:
CUTOVER = None

if CUTOVER is None:
    # last OICP snapshot if there is OCPI data, otherwise use all data of the only source
    CUTOVER = con.execute("""
        SELECT CASE
            WHEN count(*) FILTER (WHERE source = 'ocpi') = 0 THEN 'infinity'
            WHEN count(*) FILTER (WHERE source = 'oicp') = 0 THEN '-infinity'
            ELSE strftime(max(observed_at) FILTER (WHERE source = 'oicp'), '%Y-%m-%d %H:%M:%S+00')
        END
        FROM snapshots
    """).fetchone()[0]
print("Cutover:", CUTOVER)

_ = con.execute(f"""
    CREATE OR REPLACE VIEW intervals AS
    SELECT * REPLACE (least(valid_to, TIMESTAMPTZ '{CUTOVER}') AS valid_to)
    FROM intervals_raw
    WHERE source = 'oicp' AND valid_from < TIMESTAMPTZ '{CUTOVER}'
    UNION ALL
    SELECT * REPLACE (greatest(valid_from, TIMESTAMPTZ '{CUTOVER}') AS valid_from)
    FROM intervals_raw
    WHERE source = 'ocpi' AND valid_to > TIMESTAMPTZ '{CUTOVER}'
""")

## Coverage

Number of EVSEs per snapshot and source. Drops show incomplete source files, gaps in the line show
periods without snapshots (scraper not running or file unchanged). Expect a jump at the switch to OCPI,
because not all operators are migrated yet.

In [ ]:
coverage = con.sql("""
    SELECT date_trunc('hour', observed_at) AS hour, source,
           avg(evse_count) AS evses, count(*) AS snapshots
    FROM snapshots GROUP BY ALL ORDER BY hour
""").df()
ax = coverage.pivot(index="hour", columns="source", values="evses").plot(figsize=(12, 4), title="EVSEs per snapshot")
ax.set_ylabel("EVSEs");

## Status distribution over time

Status of all EVSEs sampled every hour (range join of an hourly grid with the intervals).

In [ ]:
hourly = con.sql("""
    WITH grid AS (
        SELECT unnest(generate_series(
            date_trunc('hour', (SELECT min(valid_from) FROM intervals)) + INTERVAL 1 HOUR,
            (SELECT max(valid_to) FROM intervals),
            INTERVAL 1 HOUR)) AS t
    )
    SELECT t, status, count(*) AS evses
    FROM grid JOIN intervals i ON i.valid_from <= t AND t < i.valid_to
    GROUP BY ALL ORDER BY t
""").df()
pivot = hourly.pivot(index="t", columns="status", values="evses").fillna(0)
ax = pivot.plot.area(figsize=(12, 5), title="EVSEs by status (hourly)", linewidth=0)
ax.set_ylabel("EVSEs");

In [ ]:
# share of occupied EVSEs among EVSEs with a known status
known = pivot.get("AVAILABLE", 0) + pivot.get("OCCUPIED", 0)
ax = (pivot.get("OCCUPIED", 0) / known * 100).plot(figsize=(12, 4), title="Occupancy rate (%)")
ax.set_ylabel("% occupied");

## Occupancy by weekday and hour

Time-weighted share of `OCCUPIED` among `AVAILABLE` + `OCCUPIED`, based on the hourly samples (local time Europe/Zurich).

In [ ]:
con.sql("""
    WITH grid AS (
        SELECT unnest(generate_series(
            date_trunc('hour', (SELECT min(valid_from) FROM intervals)) + INTERVAL 1 HOUR,
            (SELECT max(valid_to) FROM intervals),
            INTERVAL 1 HOUR)) AS t
    ), samples AS (
        SELECT timezone('Europe/Zurich', t) AS local_t, status
        FROM grid JOIN intervals i ON i.valid_from <= t AND t < i.valid_to
        WHERE status IN ('AVAILABLE', 'OCCUPIED')
    )
    SELECT hour(local_t) AS hour,
           round(100 * avg((status = 'OCCUPIED')::INT) FILTER (WHERE isodow(local_t) <= 5), 1) AS weekday,
           round(100 * avg((status = 'OCCUPIED')::INT) FILTER (WHERE isodow(local_t) > 5), 1) AS weekend
    FROM samples GROUP BY ALL ORDER BY hour
""").df().set_index("hour").plot(figsize=(12, 4), title="Occupancy rate by hour (%)");

## Operators

Time-weighted status shares per operator (top 15 by number of EVSEs).

In [ ]:
con.sql("""
    SELECT operator_id,
           count(DISTINCT evse_id) AS evses,
           round(100 * sum(epoch(valid_to - valid_from)) FILTER (WHERE status = 'OCCUPIED') / sum(epoch(valid_to - valid_from)), 1) AS pct_occupied,
           round(100 * sum(epoch(valid_to - valid_from)) FILTER (WHERE status = 'OUT_OF_SERVICE') / sum(epoch(valid_to - valid_from)), 1) AS pct_out_of_service,
           round(100 * sum(epoch(valid_to - valid_from)) FILTER (WHERE status = 'UNKNOWN') / sum(epoch(valid_to - valid_from)), 1) AS pct_unknown
    FROM intervals GROUP BY ALL ORDER BY evses DESC LIMIT 15
""").df()

## Charging sessions

Duration of `OCCUPIED` intervals. Adjacent intervals with the same status (cut at month boundaries or the cutover)
are merged first. The resolution is limited by the scraping interval (~8–15 min).

In [ ]:
con.sql("""
    WITH ordered AS (
        SELECT *, lag(status) OVER w AS prev_status, lag(valid_to) OVER w AS prev_to
        FROM intervals WINDOW w AS (PARTITION BY evse_id ORDER BY valid_from)
    ), grouped AS (
        SELECT *, sum(CASE WHEN status = prev_status AND valid_from = prev_to THEN 0 ELSE 1 END)
                  OVER (PARTITION BY evse_id ORDER BY valid_from) AS grp
        FROM ordered
    ), sessions AS (
        SELECT evse_id, min(valid_from) AS start, max(valid_to) AS stop
        FROM grouped WHERE status = 'OCCUPIED' GROUP BY evse_id, grp
    )
    SELECT count(*) AS sessions,
           round(quantile_cont(epoch(stop - start) / 60, 0.25)) AS p25_minutes,
           round(median(epoch(stop - start) / 60)) AS median_minutes,
           round(quantile_cont(epoch(stop - start) / 60, 0.75)) AS p75_minutes
    FROM sessions
""").df()

## History of a single EVSE

In [ ]:
EVSE_ID = con.execute("SELECT evse_id FROM intervals GROUP BY evse_id ORDER BY count(*) DESC LIMIT 1").fetchone()[0]
con.execute("""
    SELECT source, status, status_raw, valid_from, valid_to, valid_to - valid_from AS duration
    FROM intervals WHERE evse_id = $evse ORDER BY valid_from
""", {"evse": EVSE_ID}).df()

## Comparison of OICP and OCPI during the overlap

For every OCPI snapshot during the overlap, the status of each EVSE in both sources is compared.
This shows how well the status mapping fits and helps to choose `CUTOVER`.

In [ ]:
overlap = con.sql("""
    WITH t AS (
        SELECT observed_at FROM snapshots
        WHERE source = 'ocpi'
          AND observed_at <= (SELECT max(observed_at) FROM snapshots WHERE source = 'oicp')
    )
    SELECT o.status_raw AS oicp, n.status_raw AS ocpi, o.status = n.status AS same_status, count(*) AS n
    FROM t
    JOIN intervals_raw o ON o.source = 'oicp' AND o.valid_from <= t.observed_at AND t.observed_at < o.valid_to
    JOIN intervals_raw n ON n.source = 'ocpi' AND n.evse_id = o.evse_id
                        AND n.valid_from <= t.observed_at AND t.observed_at < n.valid_to
    GROUP BY ALL ORDER BY n DESC
""").df()
if len(overlap):
    print(f"Agreement: {overlap[overlap.same_status].n.sum() / overlap.n.sum():.1%}")
    display(overlap.pivot_table(index="oicp", columns="ocpi", values="n", aggfunc="sum", fill_value=0))
else:
    print("No overlap between OICP and OCPI in the loaded Parquet files")